In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
import glob, os
from sklearn.manifold import TSNE
from sklearn.preprocessing import minmax_scale
from sklearn.manifold import SpectralEmbedding
import glob

In [3]:
def calc_price_from_tick(df):
    diff = abs(df.diff())
    min_diff = np.nanmin(diff.where(lambda x: x > 0))
    n_ticks = (diff / min_diff).round()
    return 0.01 / np.nanmean(diff / n_ticks)


def calculate_custom_wap_denormalized(df):
    """
    Groups by time_id to:
    1. Select the absolute best bid/ask across Level 1 and 2 for every row.
    2. Estimate the price multiplier (denormalization) based on minimum tick.
    3. Calculate the row-level WAP in real currency terms.
    """
    
    # Get tick-based multiplier per time_id
    price_cols = ['bid_price1', 'ask_price1', 'bid_price2', 'ask_price2']
    multipliers = df.groupby('time_id')[price_cols].apply(calc_price_from_tick)
    df = df.merge(multipliers.to_frame('multiplier'), left_on='time_id', right_index=True)
    
    # Apply multiplier to all price columns
    for col in price_cols:
        df[col] = df[col] * df['multiplier']

    df.drop(columns=['multiplier'], inplace=True)  # drop if not needed later
    
    df=add_wap_at_zero(df)
    
    return df

def calculate_bid_ask_spread(df):
    """
    For test data: calculate bid-ask spread for every row.
    Computes best_bid_price and best_ask_price from raw price columns,
    then adds spread values as 'bid_ask_spread' column to the dataframe.
    """
    df = df.copy()
    
    # compute best bid/ask prices from raw columns for all rows
    df['best_bid_price'] = np.maximum(df['bid_price1'], df['bid_price2'])
    df['best_ask_price'] = np.minimum(df['ask_price1'], df['ask_price2'])
    
    # calculate bid-ask spread for all rows
    df['bid_ask_spread'] = (df['best_ask_price']/df['best_bid_price']) - 1
    
    # drop intermediate columns if not needed later
    df.drop(columns=['best_bid_price', 'best_ask_price'], inplace=True)   
    return df

def calculate_volume(df):
    """
    For test data: calculate total volume for every row.
    Computes best_bid_size and best_ask_size from raw size columns,
    then adds total volume values as 'total_volume' column to the dataframe.
    """
    df = df.copy()
    df['total_volume'] = df[['bid_size1', 'bid_size2', 'ask_size1', 'ask_size2']].sum(axis=1)
    
    # drop intermediate columns if not needed later
    return df
def calculate_depth_imbalance(df):
    """
    Depth imbalance measures the relative asymmetry between bid and ask liquidity.
    Ranges from -1 (all ask-side) to +1 (all bid-side).
    Uses total bid and ask depth across both levels.
    """
    df = df.copy()
    total_bid = df['bid_size1'] + df['bid_size2']
    total_ask = df['ask_size1'] + df['ask_size2']
    df['depth_imbalance'] = (total_bid - total_ask) / (total_bid + total_ask)
    return df

def add_wap_at_zero(df):
    """
    For test data: calculate WAP for every row.
    Computes best_bid_price/size and best_ask_price/size from raw price/size columns,
    then adds WAP values as 'multiplier' column to the dataframe.
    """
    df = df.copy()
    
    # compute best bid/ask prices and sizes from raw columns for all rows
    df['best_bid_price'] = np.maximum(df['bid_price1'], df['bid_price2'])
    df['best_ask_price'] = np.minimum(df['ask_price1'], df['ask_price2'])
    
    # best_bid_size is the size at the best (highest) bid price
    df['best_bid_size'] = np.where(df['bid_price1'] >= df['bid_price2'], 
                                   df['bid_size1'], df['bid_size2'])
    
    # best_ask_size is the size at the best (lowest) ask price
    df['best_ask_size'] = np.where(df['ask_price1'] <= df['ask_price2'], 
                                   df['ask_size1'], df['ask_size2'])
    
    # calculate WAP for all rows
    df['wap'] = (df['best_bid_price'] * df['best_ask_size'] + 
                        df['best_ask_price'] * df['best_bid_size']) / (
                        df['best_bid_size'] + df['best_ask_size'])
    df.drop(columns=['best_bid_price', 'best_ask_price', 'best_bid_size', 'best_ask_size'], inplace=True)   
    return df



In [ ]:
data_dir = "Optiver\individual_book_train"
out_dir = "Optiver\individual_book_train_denorm"
os.makedirs(out_dir, exist_ok=True)

KEEP_COLS = ['time_id', 'seconds_in_bucket', 'wap', 'bid_ask_spread', 'total_volume', 'depth_imbalance']
def fill_missing_seconds_in_bucket(df):
    # create a complete index of all (time_id, seconds) combinations
    time_ids = df['time_id'].unique()
    full_index = pd.MultiIndex.from_product(
        [time_ids, range(600)], 
        names=['time_id', 'seconds_in_bucket']
    )
    filled = (
        df.set_index(['time_id', 'seconds_in_bucket'])
          .reindex(full_index)
          .groupby(level='time_id')
          .ffill()
          .groupby(level='time_id')
          .bfill()
          .reset_index()
    )
    return filled[KEEP_COLS]


def process_file(path, out_dir):
    fname = os.path.basename(path)
    out_path = os.path.join(out_dir, fname)
    try:
        df = pd.read_csv(path)
        df = calculate_custom_wap_denormalized(df)
        df = calculate_bid_ask_spread(df)
        df = calculate_volume(df)
        df = calculate_depth_imbalance(df)
        df = df[KEEP_COLS]
        df = fill_missing_seconds_in_bucket(df)
        df.to_csv(out_path, index=False)
        return (fname, 'ok')
    except Exception as e:
        return (fname, f'error: {e}')

files = glob.glob(os.path.join(data_dir, "stock_*.csv"))
print(f"Found {len(files)} files to process")

# Choose number of workers: -1 uses all CPUs. If I/O bound, reduce this.
n_jobs = -1
print(f"Found {len(files)} files to process")

results = Parallel(n_jobs=-1, verbose=5)(
    delayed(process_file)(p, out_dir) for p in files
)

oks  = [r for r in results if r[1] == 'ok']
errs = [r for r in results if r[1] != 'ok']
print(f"Completed: {len(oks)}; Failed: {len(errs)}")
if errs:
    print("Failures (first 10):")
    for fn, msg in errs[:10]:
        print('-', fn, msg)

<>:1: SyntaxWarning: invalid escape sequence '\i'
<>:2: SyntaxWarning: invalid escape sequence '\i'
<>:1: SyntaxWarning: invalid escape sequence '\i'
<>:2: SyntaxWarning: invalid escape sequence '\i'
C:\Users\Hp\AppData\Local\Temp\ipykernel_35488\3381998223.py:1: SyntaxWarning: invalid escape sequence '\i'
  data_dir = "Optiver\individual_book_train"
C:\Users\Hp\AppData\Local\Temp\ipykernel_35488\3381998223.py:2: SyntaxWarning: invalid escape sequence '\i'
  out_dir = "Optiver\individual_book_train_denorm"
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


Found 112 files to process
Found 112 files to process


[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed: 11.7min


In [1]:
import glob
files = glob.glob(r"Optiver\individual_book_train_denorm\stock_*.csv")
print(f"Files in denorm folder: {len(files)}")

# spot check one file
import pandas as pd
df = pd.read_csv(files[0])
print(df.shape)
print(df.columns.tolist())
print(df.head())

Files in denorm folder: 112
(2298000, 6)
['time_id', 'seconds_in_bucket', 'wap', 'bid_ask_spread', 'total_volume', 'depth_imbalance']
   time_id  seconds_in_bucket         wap  bid_ask_spread  total_volume  \
0        5                  0  193.649150        0.000878         331.0   
1        5                  1  193.651874        0.000878         205.0   
2        5                  2  193.651874        0.000878         205.0   
3        5                  3  193.651874        0.000878         205.0   
4        5                  4  193.651874        0.000878         205.0   

   depth_imbalance  
0        -0.969789  
1        -0.951220  
2        -0.951220  
3        -0.951220  
4        -0.951220  


In [ ]:


# Process a new multi-stock file and reshape to wide WAP matrix
new_input_path = r"Data\Optiver\new_stock_file.csv"
new_output_path = r"Data\Optiver\new_stock_file_wap_matrix.csv"


def fill_missing_seconds_in_bucket_by_stock(df):
    """Ensure every stock/time_id has rows for all 0..599 seconds_in_bucket."""
    df = df.sort_values(['stock_id', 'time_id', 'seconds_in_bucket'])
    filled = (
        df.groupby(['stock_id', 'time_id'], group_keys=False)
          .apply(lambda g: g.set_index('seconds_in_bucket')
                         .reindex(range(600))
                         .ffill()
                         .bfill()
                         .assign(stock_id=g['stock_id'].iloc[0], time_id=g['time_id'].iloc[0]))
          .reset_index()
    )
    return filled[['stock_id', 'time_id', 'seconds_in_bucket', 'wap', 'bid_ask_spread', 'total_volume']]


# Read the new file and apply the same feature pipeline.
df_new = pd.read_csv(new_input_path)

df_new = calculate_custom_wap_denormalized(df_new)
df_new = calculate_bid_ask_spread(df_new)
df_new = calculate_volume(df_new)
df_new = df_new[['stock_id', 'time_id', 'seconds_in_bucket', 'wap', 'bid_ask_spread', 'total_volume']]
df_new = fill_missing_seconds_in_bucket_by_stock(df_new)
df_new['stock_id'] = df_new['stock_id'].astype(str)

# Pivot to the requested wide format: time_id, seconds_in_bucket, stock columns with wap values.
df_wide = (
    df_new.pivot(index=['time_id', 'seconds_in_bucket'], columns='stock_id', values='wap')
      .reset_index()
)

# Ensure stock columns are ordered numerically if possible.
stock_cols = [c for c in df_wide.columns if c not in ['time_id', 'seconds_in_bucket']]
try:
    stock_cols_sorted = sorted(stock_cols, key=lambda x: int(x))
except ValueError:
    stock_cols_sorted = sorted(stock_cols)

output_cols = ['time_id', 'seconds_in_bucket'] + stock_cols_sorted
df_wide = df_wide[output_cols]
df_wide.to_csv(new_output_path, index=False)

print(f"Saved wide WAP matrix with {len(stock_cols_sorted)} stock columns to: {new_output_path}")

In [ ]:

files = glob.glob(r"Data\Optiver\individual_book_train_denorm\stock_*.csv")
mult_list = []
for path in files:
    stock = os.path.basename(path).split('stock_')[-1].split('.csv')[0]
    # read seconds_in_bucket so we can index by the combination
    df_local = pd.read_csv(path, usecols=['time_id','seconds_in_bucket','wap'])
    # create a series indexed by (time_id, seconds_in_bucket)
    ser = df_local.set_index(['time_id','seconds_in_bucket'])['wap'].rename(stock)
    mult_list.append(ser)

# concat along columns; the index is now a MultiIndex of (time_id, seconds_in_bucket)

df_multiplier = pd.concat(mult_list, axis=1)
# ensure column order sorted by int value if possible
try:
    cols_sorted = sorted(df_multiplier.columns, key=lambda x: int(x))
    df_multiplier = df_multiplier[cols_sorted]
except Exception:
    pass

print(df_multiplier.head())
df_multiplier.to_csv('multiplier_allstocks_final.csv')
plt.figure(figsize=(15, 20))
ax = sns.stripplot(data=df_multiplier, orient='h', alpha=0.3, s=2, jitter=0.2,
                   order=df_multiplier.median().sort_values().index[::-1].tolist(), 
                   palette='Blues')
ax.tick_params(axis='y', which='major', labelsize=10)
plt.xlabel('price')
plt.title('Denormalized price distribution by stock')